In [ ]:
# 0. Import libraries
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from optuna.distributions import FloatDistribution, IntDistribution, CategoricalDistribution
import sys
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

sys.path.append('../src')
from functions import (
    RepeatedNestedCV,
    summarize_with_ci
)

In [ ]:
# 1. Load and preprocess data
data = pd.read_csv('../data/breast_cancer_cleaned.csv')

# 1.1 Split into X and y
X = data.drop(columns=['diagnosis']).values
y = data['diagnosis'].values

In [ ]:
# 2. Define estimators with F-test feature selection (top-k)
estimators_f = {
    'LogisticRegression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('select', SelectKBest(score_func=f_classif, k=10)),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(penalty='elasticnet', solver='saga', max_iter=10000))
    ]),
    'GaussianNB': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('select', SelectKBest(score_func=f_classif, k=10)),
        ('clf', GaussianNB())
    ]),
    'LDA': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('select', SelectKBest(score_func=f_classif, k=10)),
        ('scaler', StandardScaler()),
        ('clf', LinearDiscriminantAnalysis())
    ]),
    'SVM': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('select', SelectKBest(score_func=f_classif, k=10)),
        ('scaler', StandardScaler()),
        ('clf', SVC(probability=True))
    ]),
    'RandomForest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('select', SelectKBest(score_func=f_classif, k=10)),
        ('clf', RandomForestClassifier())
    ]),
    'LightGBM': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('select', SelectKBest(score_func=f_classif, k=10)),
        ('clf', LGBMClassifier())
    ])
}


In [ ]:
# 2.1 Define estimators with Mutual Information feature selection (top-k)
estimators_mi = {
    'LogisticRegression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('select', SelectKBest(score_func=mutual_info_classif, k=10)),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(penalty='elasticnet', solver='saga', max_iter=10000))
    ]),
    'GaussianNB': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('select', SelectKBest(score_func=mutual_info_classif, k=10)),
        ('clf', GaussianNB())
    ]),
    'LDA': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('select', SelectKBest(score_func=mutual_info_classif, k=10)),
        ('scaler', StandardScaler()),
        ('clf', LinearDiscriminantAnalysis())
    ]),
    'SVM': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('select', SelectKBest(score_func=mutual_info_classif, k=10)),
        ('scaler', StandardScaler()),
        ('clf', SVC(probability=True))
    ]),
    'RandomForest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('select', SelectKBest(score_func=mutual_info_classif, k=10)),
        ('clf', RandomForestClassifier())
    ]),
    'LightGBM': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('select', SelectKBest(score_func=mutual_info_classif, k=10)),
        ('clf', LGBMClassifier())
    ])
}


In [ ]:
# 3. Define hyperparameter search spaces for each estimator
param_grids = {
    'LogisticRegression': {
        'clf__C': FloatDistribution(1e-2, 10, log=True),
        'clf__l1_ratio': FloatDistribution(0.0, 1.0),
    },
    'GaussianNB': {
        'clf__var_smoothing': FloatDistribution(1e-9, 1e-7, log=True),
    },
    'LDA': {
        'clf__solver': CategoricalDistribution(['svd', 'lsqr', 'eigen']),
        'clf__shrinkage': CategoricalDistribution(['auto', None]),
    },
    'SVM': {
        'clf__C': FloatDistribution(1e-2, 10, log=True),
        'clf__kernel': CategoricalDistribution(['linear', 'rbf']),
    },
    'RandomForest': {
        'clf__n_estimators': IntDistribution(100, 300),
        'clf__max_depth': IntDistribution(3, 15),
        'clf__min_samples_split': IntDistribution(2, 10),
    },
    'LightGBM': {
        'clf__n_estimators': IntDistribution(100, 300),
        'clf__max_depth': IntDistribution(3, 15),
        'clf__learning_rate': FloatDistribution(0.01, 0.2),
    }
}

In [ ]:
# 4. Run nested cross-validation with F-test feature selection
nested_cv_f = RepeatedNestedCV(
    estimators=estimators_f,
    param_grids=param_grids,
    R=10,
    N=5,
    K=3,
    scoring='balanced_accuracy'
)

print("=== Nested CV: Top F-test features ===")
results_f = nested_cv_f.run(X, y)



In [ ]:
# 5. Save F-test results for later analysis
results_f['Method'] = 'F-test'
results_f.to_csv("../results/nested_cv_results_f_test.csv", index=False)


In [ ]:
# 6. Run nested cross-validation with Mutual Information feature selection
nested_cv_mi = RepeatedNestedCV(
    estimators=estimators_mi,
    param_grids=param_grids,
    R=10,
    N=5,
    K=3,
    scoring='balanced_accuracy'
)

print("\n=== Nested CV: Top Mutual Information features ===")
results_mi = nested_cv_mi.run(X, y)

In [ ]:
# 7. Save Mutual Information results for later analysis
results_mi['Method'] = 'Mutual Information'
results_mi.to_csv("../results/nested_cv_results_mutual_info.csv", index=False)

In [ ]:
# 8. Analyze and visualize metrics F test
summarize_with_ci(results_f)

In [ ]:
# 9. Analyze and visualize metrics mutual_info
summarize_with_ci(results_mi)